# Modelforge ASE calculator 
## Simple Example

This notebook demonstrates the steps required to use a potential trained using modelforge as a calculator within the atomic simulation environment (ASE). 

First let us import some various modeules from modelforge we require for reading in a checkpoint file from a trained model

In [ ]:
from modelforge.potential.potential import load_inference_model_from_checkpoint

# helper functions to load in the example model checkpoint file included in the repository
from modelforge.utils.io import get_path_string
from modelforge.ase.tests import data

# checkpoint file is saved in tests/data
checkpoint_file_path = get_path_string(data) + "/model.ckpt"
potential = load_inference_model_from_checkpoint(checkpoint_file_path, jit=False)


To use this potential in ASE, we will need to load the ModelForgeCalculator and pass it the potential on instantiation.

In [ ]:
from modelforge.ase.calculator import ModelForgeCalculator

modelforge_calculator = ModelForgeCalculator(potential)


ASE has various built-in functions to set up molecules, but also a well defined API for manually setting these up.  As part of the modelforge.ase module, we have included some helper functions that will use accept a smiles string, use rdkit to initialize the 3d coordinates, and then initialize the configuration in a format ASE understands. 

In [ ]:
from modelforge.ase.examples.helper_functions import smiles_to_ase, ase_to_rdkit

# note, optimize=True would use MMFF94 to optimize the structure within RDkit. 

smiles = "NCCCCCCO"
atoms = smiles_to_ase(smiles, optimize=False)
atoms.calc = modelforge_calculator



ASE provides a built in visualizer for viewing the 3d structure. Alternatively, we added a simple function to  `helper_functions.py` that will convert to rdkit. 

In [ ]:
from ase.visualize import view
view(atoms, viewer='x3d')

### Single point energy and force computation
To extract the potential energy and per-atom forces on the system, we can use the following syntax. This will call the `ModelForgeCalculator` instance we set above.  Note, the calculator performs all appropriate unit conversions (modelforge internally uses kJ/mol and nanometers).  The units in ASE are eV and angstrom. 

In [ ]:
# extract the energy adn forcesfrom ase.io import write, read
pe = atoms.get_potential_energy()
forces = atoms.get_forces()
print(f"potential energy: {pe}\n eV")
print(f"forces: {forces}\n eV/angstrom")

### System optimization 
ASE provides a host of functionality, such as optimization.

In [ ]:
from ase.optimize import BFGS

opt = BFGS(atoms)
opt.run(fmax=0.05)

# System simulation
We can also perform simulations. 

In [ ]:
import ase.units as ase_units
from ase.io.trajectory import Trajectory
from ase.md import Langevin
from ase.io import write, read

trajectory_name= f"{smiles}_sim.traj"
write(f'{smiles}_system.pdb', atoms)

traj = Trajectory(trajectory_name, "w", atoms)

dyn = Langevin(
    atoms,
    timestep=1.0* ase_units.fs,
    temperature_K=298,  # temperature in Kfrom ase.io import write, read
    friction=1.0/ ase_units.fs,
    trajectory=traj,
    logfile=f"{smiles}_md.log",
)
dyn.run(100)

# convert the ASE binary format to something vmd can read
traj = read(trajectory_name, ":")
write(f"{smiles}_sim.xyz", traj, format="xyz")